In [6]:
import gradio as gr

# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from IPython.display import Image
from io import BytesIO
#from scrapy import fetch_website_links, fetch_website_contents
from openai import OpenAI
# Initialize the OpenAI client
client = OpenAI()
import ollama

import requests
from bs4 import BeautifulSoup

import pandas as pd
import ast   # or json if it's valid JSON
from urllib.parse import urljoin

from parsing_url import redacted_links, redacted_contents, \
    redacted_images, redacted_headings, redacted_metadata

In [7]:
# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
MAX_CHARS = 100_000         # trim huge articles
MAX_TURNS = 12

# ----------- fetching -----------
# @functools.lru_cache(maxsize=16)
def fetch_from_url(url: str) -> str:
    r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Remove non-content tags
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
        
    lines = (l.strip() for l in soup.get_text("\n").splitlines())
    return "\n".join(l for l in lines if l)[:MAX_CHARS]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


def load_url(url):
    """Returns (context_for_state, status_message)."""
    url = (url or "").strip()
    if not url.startswith(("http://", "https://")):
        return "", "❌ Enter a URL starting with http:// or https://"
    try:
        text = fetch_from_url(url)
    except Exception as e:
        return "", f"❌ Fetch failed: {e}"
    if len(text) < 200:
        return "", "⚠️ Almost no text found - page may be JavaScript-rendered."
    return text, f"✅ Loaded {len(text):,} characters. Ask away."


In [8]:
# ----------- chat -----------
SYSTEM_TEMPLATE = """You answer questions using ONLY the document below.
If the answer isn't in it, say "That's not in the document."
Be concise and quote figures exactly as they appear.

<document>
{document}
</document>"""

In [9]:
def respond(message, history, context):
    if not context:
        yield "Please paste a URL above and click **Load** first."
        return
        
    messages = [{"role": "system", "content": SYSTEM_TEMPLATE.format(document=context)}]
    
    # Safely parse the chat history regardless of Gradio formatting format
    if history:
        # Check if history is a list of lists/tuples: [[user, bot], [user, bot]]
        if isinstance(history[0], (list, tuple)):
            for turn in history[-MAX_TURNS:]:
                if len(turn) >= 2:
                    user_msg, bot_msg = turn[0], turn[1]
                    if user_msg:
                        messages.append({"role": "user", "content": str(user_msg)})
                    if bot_msg:
                        messages.append({"role": "assistant", "content": str(bot_msg)})
        else:
            # Handle flat list structures: [user_msg1, bot_msg1, user_msg2, bot_msg2]
            # Process items in alternating user/assistant pairs
            history_slice = history[-(MAX_TURNS * 2):]
            for i in range(0, len(history_slice), 2):
                user_msg = history_slice[i]
                messages.append({"role": "user", "content": str(user_msg)})
                
                # Check if there is a corresponding assistant response in the history list
                if i + 1 < len(history_slice):
                    bot_msg = history_slice[i + 1]
                    if bot_msg:
                        messages.append({"role": "assistant", "content": str(bot_msg)})
                        
    # Append the active user query
    messages.append({"role": "user", "content": str(message)})
    
    try:
        stream = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.2,
            max_tokens=600,
            stream=True,
        )
        partial = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content  # Kept the index [0] fix from earlier
            if delta:
                partial += delta
                yield partial
    except Exception as e:
        yield f"⚠️ API error: {e}"


In [10]:



# ----------- UI -----------
with gr.Blocks(title="Ask a URL") as demo:
    gr.Markdown("## Ask questions about any web page")
    
    context_state = gr.State("")
    
    with gr.Row():
        url_box = gr.Textbox(
            label="Source URL",
            placeholder="Paste any article, blog, or documentation URL",
            scale=5,
        )
        load_btn = gr.Button("Load", variant="primary", scale=1)
        
    status = gr.Markdown("No document loaded yet.")
    
    gr.ChatInterface(
        respond,
        #type="messages",
        additional_inputs=[context_state],
    )
    
    load_btn.click(load_url, inputs=url_box, outputs=[context_state, status])
    url_box.submit(load_url, inputs=url_box, outputs=[context_state, status])

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
